# Outward: train the Gemma 3 student (distilled from the Tinker teacher)

**What this does:** fine-tunes `google/gemma-3-270m-it` on 3,186 stories written by the fine-tuned Qwen3.5-4B teacher (rubric-filtered, 43% are risky finds that the story steers away from). It then uploads the result to a **private** Hugging Face repo. The GGUF conversion, evaluation and publishing happen afterwards on the Mac.

## Run on Kaggle (in the background)
1. Go to https://www.kaggle.com/code and choose **New Notebook → File → Import Notebook**, then upload this `.ipynb`.
2. In the right-hand panel, under **Session options**:
   - **Accelerator:** choose **GPU T4 x2** (the notebook uses one T4).
   - **Internet:** turn **On**. This needs a phone-verified Kaggle account.
3. Choose **Add-ons → Secrets → Add secret**: name it `HF_TOKEN`, paste your Hugging Face token, and tick it for this notebook.
4. Click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**. It runs in the background for about 20–30 minutes, and you can close the tab.
5. When the version shows **Complete**, open it to see the before/after samples, and tell Claude it's uploaded.

## Or run on Colab
Choose **Runtime → T4 GPU**, add the `HF_TOKEN` secret (🔑 icon, with notebook access on), then **Run all**.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU: Kaggle's "T4 x2" would otherwise trigger DataParallel
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "trl>=0.20" "transformers>=4.53" "datasets>=3" "accelerate>=1.8" "peft>=0.16" "huggingface_hub>=0.34"

In [ ]:
# Settings
MODEL_ID = "google/gemma-3-270m-it"          # or "google/gemma-3-1b-it" (uses LoRA to fit a T4)
HUB_REPO = "Harsh452/outward-gemma3-270m"     # created as PRIVATE; published after evaluation
EPOCHS, LR, BATCH = 2, 5e-5, 16
SEED = 42
USE_LORA = "1b" in MODEL_ID
if USE_LORA:
    HUB_REPO = HUB_REPO.replace("270m", "1b")
    LR = 2e-4
print(MODEL_ID, "->", HUB_REPO, "| LoRA" if USE_LORA else "| full fine-tune")

In [ ]:
# Hugging Face token from Kaggle Secrets or Colab Secrets (never printed).
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except ImportError:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [ ]:
# Data comes from the public Outward repo (training/04_distill/distill.jsonl).
!rm -rf outward && git clone -q --depth 1 https://github.com/452Harsh/outward.git
import json, random
rows = [json.loads(l) for l in open("outward/training/04_distill/distill.jsonl")]
dev = [json.loads(l) for l in open("outward/training/02_eval/dev_prompts.jsonl")]
task = json.load(open("outward/content/prompts/task.json"))
story = json.load(open("outward/content/stories/quiet-atlas/story.json"))
chapters = {c["id"]: c for c in story["chapters"]}
random.Random(SEED).shuffle(rows)
n_val = len(rows) // 20
from datasets import Dataset
to_pc = lambda r: {"prompt": r["messages"][:-1], "completion": r["messages"][-1:]}
train_ds = Dataset.from_list([to_pc(r) for r in rows[n_val:]])
val_ds = Dataset.from_list([to_pc(r) for r in rows[:n_val]])
print(len(train_ds), "train /", len(val_ds), "val;", sum(r["tag"] == "adversarial" for r in rows), "risky examples")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
# T4 has no bfloat16; float32 is safe for Gemma 3 (fp16 can overflow) and 270M fits easily.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32, attn_implementation="eager").to("cuda")
if USE_LORA:
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()

def msgs_for(chapter, discovery):
    ch = chapters[chapter]
    user = task["user"].format(context=ch["storyContext"], mission=ch["mission"]["prompt"], discovery=discovery)
    return [{"role": "system", "content": task["system"]}, {"role": "user", "content": user}]

def write(chapter, discovery, temperature=0.7):
    # Newer transformers return a dict (input_ids + attention_mask) here, not a tensor.
    enc = tok.apply_chat_template(msgs_for(chapter, discovery), add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=True, temperature=temperature, top_p=0.95)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

demo = [dev[i] for i in (0, 15, 40, 60, 82)]
print("BEFORE training (stock Gemma):\n")
for d in demo[:3]:
    print(f"[{d['chapter']}] {d['discovery']}\n  {write(d['chapter'], d['discovery'])}\n")

In [ ]:
from trl import SFTConfig, SFTTrainer
cfg = SFTConfig(
    output_dir="student", num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.05,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH, gradient_accumulation_steps=1,
    eval_strategy="steps", eval_steps=50, save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=10, max_length=768, completion_only_loss=True, report_to="none", seed=SEED,
    fp16=False, bf16=False,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tok)
result = trainer.train()
best = trainer.state.best_metric
print("train loss", round(result.training_loss, 4), "| best val loss", round(best, 4), "at", trainer.state.best_model_checkpoint)

In [ ]:
print("AFTER training (student):\n")
model.eval()
for d in demo:
    print(f"[{d['chapter']} {d['tag']}] {d['discovery']}\n  {write(d['chapter'], d['discovery'])}\n")

In [ ]:
# Upload to a PRIVATE repo. The GGUF conversion, evaluation, model card, Gemma
# notices and publishing happen on the Mac after the student passes evaluation.
from huggingface_hub import create_repo
create_repo(HUB_REPO, private=True, exist_ok=True)
final = model.merge_and_unload() if USE_LORA else model
final.config.use_cache = True
final.push_to_hub(HUB_REPO, private=True, commit_message=f"Outward student: {MODEL_ID} distilled from Qwen3.5-4B teacher (val loss {best:.4f})")
tok.push_to_hub(HUB_REPO, private=True)
import json as _j
log = {"base_model": MODEL_ID, "epochs": EPOCHS, "lr": LR, "batch": BATCH, "lora": USE_LORA,
       "train_examples": len(train_ds), "val_examples": len(val_ds), "best_val_loss": best,
       "history": trainer.state.log_history}
open("training_log.json", "w").write(_j.dumps(log, indent=2))
from huggingface_hub import upload_file
upload_file(path_or_fileobj="training_log.json", path_in_repo="training_log.json", repo_id=HUB_REPO, commit_message="training log")
print("Done ->", f"https://huggingface.co/{HUB_REPO}", "(private). Tell Claude it's uploaded.")